## Few-shot Test on CRC100K

In [6]:
import os
import argparse
import pandas as pd
import numpy as np
import torch
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import sys
sys.argv = ['']  # Prevents argparse conflicts
import argparse

# CONCH imports
from conch.open_clip_custom import create_model_from_pretrained
from conch.downstream.zeroshot_path import zero_shot_classifier


# ---------------------------------------------------------------------------
# Evaluation helpers
# ---------------------------------------------------------------------------

def evaluate_classifier(y_true, y_pred, labels, title, results_dir):
    acc = accuracy_score(y_true, y_pred)
    b_acc = balanced_accuracy_score(y_true, y_pred)

    print(f"\n===== {title.upper()} RESULTS =====")
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced Accuracy: {b_acc:.4f}")

    report_dict = classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()
    print("\nClassification Report:")
    print(report_df.round(3))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Confusion Matrix – {title}", fontsize=13, fontweight="bold")
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout()
    os.makedirs(results_dir, exist_ok=True)
    save_path = os.path.join(results_dir, f"{title.replace(' ', '_').lower()}_cm.png")
    plt.savefig(save_path, dpi=200)
    plt.close()
    print(f"  Confusion matrix saved → {save_path}")

    return {"title": title, "accuracy": acc, "balanced_accuracy": b_acc, "report": report_dict}


def plot_comparison_bar(all_results, results_dir):
    """Side-by-side bar chart comparing all classifiers."""
    titles = [r["title"] for r in all_results]
    accs   = [r["accuracy"] for r in all_results]
    b_accs = [r["balanced_accuracy"] for r in all_results]

    x = np.arange(len(titles))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(10, len(titles) * 2.5), 6))
    bars1 = ax.bar(x - width / 2, accs,   width, label="Accuracy",          color="#4C8BE3")
    bars2 = ax.bar(x + width / 2, b_accs, width, label="Balanced Accuracy", color="#E3844C")

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_ylim(0, 1.12)
    ax.set_xticks(x)
    ax.set_xticklabels(titles, rotation=15, ha="right", fontsize=9)
    ax.set_ylabel("Score")
    ax.set_title("Classifier Comparison: Zero-Shot vs Few-Shot Methods", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    save_path = os.path.join(results_dir, "classifier_comparison.png")
    plt.savefig(save_path, dpi=200)
    plt.close()
    print(f"\nComparison bar chart saved → {save_path}")


# ---------------------------------------------------------------------------
# Dataset building from class subfolders
# (reads images directly from  <image_folder>/<class_name>/<image>.*)
# ---------------------------------------------------------------------------

def build_df_from_subfolders(image_folder, classes_list):
    """
    Walk <image_folder>/<class_name>/ and return a DataFrame with
    columns [patch_name, label, full_path].
    """
    records = []
    extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    for cls in classes_list:
        cls_dir = os.path.join(image_folder, cls)
        if not os.path.isdir(cls_dir):
            print(f"  [WARNING] Class folder not found: {cls_dir}")
            continue
        for fname in os.listdir(cls_dir):
            if Path(fname).suffix.lower() in extensions:
                records.append({
                    "patch_name": fname,
                    "label":      cls,
                    "full_path":  os.path.join(cls_dir, fname),
                })
    df = pd.DataFrame(records)
    print(f"  Found {len(df)} total patches across {df['label'].nunique()} classes.")
    return df


# ---------------------------------------------------------------------------
# Train / test split: exactly N_TRAIN patches for training, rest for test
# ---------------------------------------------------------------------------

def split_train_test(df, classes_list, n_train_total, random_seed=42):
    """
    Sample n_train_total patches in a class-balanced way for training.
    Everything else goes to the test set.
    """
    rng = np.random.default_rng(random_seed)
    n_classes = len(classes_list)
    per_class  = n_train_total // n_classes          # floor; leftover ignored

    train_indices = []
    for cls in classes_list:
        idx = df.index[df["label"] == cls].tolist()
        rng.shuffle(idx)
        take = min(per_class, len(idx))
        train_indices.extend(idx[:take])

    train_mask = df.index.isin(train_indices)
    train_df   = df[train_mask].reset_index(drop=True)
    test_df    = df[~train_mask].reset_index(drop=True)

    print(f"\n  Train set: {len(train_df)} patches  ({per_class} per class)")
    print(f"  Test  set: {len(test_df)} patches")
    per_cls_counts = train_df["label"].value_counts().to_dict()
    print(f"  Train class distribution: {per_cls_counts}")
    return train_df, test_df


# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------

def extract_features_from_df(df, model, preprocess, device, desc="Extracting features"):
    """
    Expects df to have a 'full_path' and 'label' column.
    Returns (torch.Tensor [N, D], np.ndarray [N]).
    """
    features, labels = [], []
    with torch.inference_mode():
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
            img_path = row["full_path"]
            if not os.path.exists(img_path):
                continue
            try:
                image = Image.open(img_path).convert("RGB")
                tensor = preprocess(image).unsqueeze(0).to(device)
                feat = model.encode_image(tensor)
                feat = feat / feat.norm(dim=-1, keepdim=True)
                features.append(feat.cpu().squeeze(0))
                labels.append(row["label"])
            except Exception as e:
                print(f"  [WARN] Skipping {img_path}: {e}")

    if not features:
        return None, None
    return torch.stack(features), np.array(labels)


# ---------------------------------------------------------------------------
# Few-shot classifier implementations
# ---------------------------------------------------------------------------

def compute_prototypes(X, y, classes):
    """Mean embedding per class, L2-normalised (ProtoNet / Nearest Centroid)."""
    prototypes = []
    for cls in classes:
        mask = y == cls
        if not np.any(mask):
            prototypes.append(torch.zeros(X.shape[1]))
            continue
        cls_feat = X[mask].mean(dim=0)
        cls_feat = cls_feat / cls_feat.norm(dim=-1, keepdim=True)
        prototypes.append(cls_feat)
    return torch.stack(prototypes)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    args = argparse.Namespace(
        prompt_file   = r"config_files\CRC100K_prompts_all_per_class.json",
        checkpoint    = r"checkpoints\conch\pytorch_model.bin",
        image_folder  = r"D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM",
        results_dir   = r"Results\Few_Shot_Evaluation" + f"\{n_train}_trainset",
        zs_weights    = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\classifier_weights\Conch_zeroshot_weights_CRC100K.pt",
        n_train       = 180, # Total number of training patches (e.g. 900 for 9 classes × 100 each)
        random_seed   = 42,
        train_cache   = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\Few_shot_features\conch_train_features.pt",
        test_cache    = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\Few_shot_features\conch_test_features.pt",
        force_extract = False,
    )

    # ------------------------------------------------------------------ #
    # 0. Device & prompts
    # ------------------------------------------------------------------ #
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    with open(args.prompt_file) as f:
        prompt_data = json.load(f)["0"]
    classnames  = prompt_data["classnames"]
    templates   = prompt_data["templates"]

    classes_list    = list(classnames.keys())           # preserves JSON order
    classnames_text = [classnames[c] for c in classes_list]
    print(f"Loaded {len(classes_list)} classes: {classes_list}")

    # ------------------------------------------------------------------ #
    # 1. Build full dataset DataFrame from subfolder structure
    # ------------------------------------------------------------------ #
    print("\nScanning image folder...")
    full_df = build_df_from_subfolders(args.image_folder, classes_list)

    if full_df.empty:
        print("No images found. Check --image_folder and class names in the prompt file.")
        return

    # ------------------------------------------------------------------ #
    # 2. Train / test split
    # ------------------------------------------------------------------ #
    train_df, test_df = split_train_test(
        full_df, classes_list, n_train_total=args.n_train, random_seed=args.random_seed
    )

    # ------------------------------------------------------------------ #
    # 3. Load model
    # ------------------------------------------------------------------ #
    print("\nLoading CONCH model...")
    model, preprocess = create_model_from_pretrained(
        model_cfg="conch_ViT-B-16",
        checkpoint_path=args.checkpoint,
        device=device,
        force_image_size=224,
    )
    model.eval()

    # ------------------------------------------------------------------ #
    # 4. Extract / cache features
    # ------------------------------------------------------------------ #
    # --- Training features ---
    if not args.force_extract and os.path.exists(args.train_cache):
        print(f"\nLoading cached TRAIN features from {args.train_cache} ...")
        cache = torch.load(args.train_cache, weights_only=False)
        X_train_t, y_train = cache["features"], cache["labels"]
    else:
        print("\nExtracting TRAIN features...")
        X_train_t, y_train = extract_features_from_df(
            train_df, model, preprocess, device, desc="Train"
        )
        torch.save({"features": X_train_t, "labels": y_train}, args.train_cache)
        print(f"  Cached → {args.train_cache}")

    # --- Test features ---
    if not args.force_extract and os.path.exists(args.test_cache):
        print(f"\nLoading cached TEST features from {args.test_cache} ...")
        cache = torch.load(args.test_cache, weights_only=False)
        X_test_t, y_test = cache["features"], cache["labels"]
    else:
        print("\nExtracting TEST features...")
        X_test_t, y_test = extract_features_from_df(
            test_df, model, preprocess, device, desc="Test"
        )
        torch.save({"features": X_test_t, "labels": y_test}, args.test_cache)
        print(f"  Cached → {args.test_cache}")

    X_train = X_train_t.numpy()
    X_test  = X_test_t.numpy()

    print(f"\nFinal split → Train: {len(y_train)} | Test: {len(y_test)}")

    all_results = []
    os.makedirs(args.results_dir, exist_ok=True)

    # ================================================================== #
    # CLASSIFIER A: ZERO-SHOT
    # ================================================================== #
    print("\n" + "="*60)
    print("CLASSIFIER A: ZERO-SHOT (text prompts only, no training patches)")
    print("="*60)

    if os.path.exists(args.zs_weights):
        print(f"Loading precomputed zero-shot weights from {args.zs_weights} ...")
        zeroshot_weights = torch.load(args.zs_weights, map_location=device, weights_only=False)
    else:
        try:
            print("Generating zero-shot classifier weights from text prompts...")
            zeroshot_weights = zero_shot_classifier(
                model, classnames_text, templates, device=device
            )
        except Exception as e:
            print(f"  [ERROR] Could not generate zero-shot weights: {e}")
            zeroshot_weights = None

    zs_logits = None
    if zeroshot_weights is not None:
        zs_logits = (100.0 * X_test_t.to(device) @ zeroshot_weights).cpu()
        zs_preds  = [classes_list[i] for i in zs_logits.argmax(dim=-1)]
        res = evaluate_classifier(y_test, zs_preds, classes_list, "Zero-Shot", args.results_dir)
        all_results.append(res)
    else:
        print("Skipping Zero-Shot (weights unavailable).")

    # ================================================================== #
    # CLASSIFIER B: FEW-SHOT NEAREST CENTROID (ProtoNet)
    # ================================================================== #
    print("\n" + "="*60)
    print(f"CLASSIFIER B: FEW-SHOT NEAREST CENTROID  (trained on {len(y_train)} patches)")
    print("="*60)

    prototypes     = compute_prototypes(X_train_t, y_train, classes_list).to(device)
    proto_logits   = (100.0 * X_test_t.to(device) @ prototypes.T).cpu()
    proto_preds    = [classes_list[i] for i in proto_logits.argmax(dim=-1)]
    res = evaluate_classifier(
        y_test, proto_preds, classes_list,
        f"Few-Shot Nearest Centroid ({len(y_train)} train patches)", args.results_dir
    )
    all_results.append(res)

    # ================================================================== #
    # CLASSIFIER C: FEW-SHOT LOGISTIC REGRESSION (Linear Probe)
    # ================================================================== #
    print("\n" + "="*60)
    print(f"CLASSIFIER C: FEW-SHOT LOGISTIC REGRESSION  (trained on {len(y_train)} patches)")
    print("="*60)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=args.random_seed)
    clf.fit(X_train, y_train)
    lr_preds = clf.predict(X_test)
    res = evaluate_classifier(
        y_test, lr_preds, classes_list,
        f"Few-Shot Logistic Regression ({len(y_train)} train patches)", args.results_dir
    )
    all_results.append(res)

    import joblib
    clf_path = os.path.join(args.results_dir, "few_shot_logistic_regression.pkl")
    joblib.dump(clf, clf_path)
    print(f"  Saved logistic regression model → {clf_path}")

    # ================================================================== #
    # CLASSIFIER D: ENSEMBLE (Zero-Shot + ProtoNet, TIP-Adapter style)
    # ================================================================== #
    if zs_logits is not None:
        print("\n" + "="*60)
        print("CLASSIFIER D: ENSEMBLE  (α·Zero-Shot + (1-α)·ProtoNet)")
        print("="*60)
        alpha     = 0.5
        zs_probs  = torch.nn.functional.softmax(zs_logits,   dim=-1)
        fs_probs  = torch.nn.functional.softmax(proto_logits, dim=-1)
        ens_probs = alpha * zs_probs + (1 - alpha) * fs_probs
        ens_preds = [classes_list[i] for i in ens_probs.argmax(dim=-1)]
        res = evaluate_classifier(
            y_test, ens_preds, classes_list,
            "Ensemble (Zero-Shot + ProtoNet)", args.results_dir
        )
        all_results.append(res)
    else:
        print("Skipping Ensemble (zero-shot weights unavailable).")

    # ================================================================== #
    # Summary table + comparison chart
    # ================================================================== #
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    summary_rows = []
    for r in all_results:
        summary_rows.append({
            "Classifier":         r["title"],
            "Accuracy":           f"{r['accuracy']:.4f}",
            "Balanced Accuracy":  f"{r['balanced_accuracy']:.4f}",
        })
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))

    summary_csv = os.path.join(args.results_dir, "summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"\nSummary saved → {summary_csv}")

    if len(all_results) > 1:
        plot_comparison_bar(all_results, args.results_dir)

    print(f"\nAll results saved in: {args.results_dir}")
    print("Done!")


if __name__ == "__main__":
    main()

Using device: cuda:0
Loaded 9 classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']

Scanning image folder...
  Found 100000 total patches across 9 classes.

  Train set: 180 patches  (20 per class)
  Test  set: 99820 patches
  Train class distribution: {'ADI': 20, 'BACK': 20, 'DEB': 20, 'LYM': 20, 'MUC': 20, 'MUS': 20, 'NORM': 20, 'STR': 20, 'TUM': 20}

Loading CONCH model...

Extracting TRAIN features...


Train: 100%|██████████| 180/180 [00:00<00:00, 195.49it/s]


  Cached → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\Few_shot_features\conch_train_features.pt

Extracting TEST features...


Test: 100%|██████████| 99820/99820 [25:01<00:00, 66.47it/s] 


  Cached → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\Few_shot_features\conch_test_features.pt

Final split → Train: 180 | Test: 99820

CLASSIFIER A: ZERO-SHOT (text prompts only, no training patches)
Loading precomputed zero-shot weights from D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\classifier_weights\Conch_zeroshot_weights_CRC100K.pt ...

===== ZERO-SHOT RESULTS =====
Accuracy:          0.7484
Balanced Accuracy: 0.7487

Classification Report:
              precision  recall  f1-score    support
ADI               0.928   0.971     0.949  10387.000
BACK              0.605   0.751     0.670  10546.000
DEB               0.759   0.901     0.824  11492.000
LYM               0.811   0.997     0.895  11537.000
MUC               0.702   0.848     0.769   8876.000
MUS               0.772   0.914     0.837  13516.000
NORM              0.914   0.525     0.667   8743.000
STR            

## Configuration

In [1]:
import os
import json
from pathlib import Path
from PIL import Image
import torch
from conch.open_clip_custom import create_model_from_pretrained, get_tokenizer
from conch.downstream.zeroshot_path import zero_shot_classifier
from torchvision import transforms

# ========== CONFIG ==========
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\checkpoints\CONCH\pytorch_model.bin"
prompt_file  = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\config_files\CRC100K_prompts_all_per_class.json"
force_image_size = 224

c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model, preprocess = create_model_from_pretrained(
    model_cfg='conch_ViT-B-16',
    checkpoint_path=checkpoint_path,
    device=device,
    force_image_size=force_image_size
)
_ = model.eval()

### Stand-alone code to train and save a few shot logistic regression model on CRC100K

In [3]:
import numpy as np
import torch
import json
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
import joblib

# ========== CONFIG ==========
n_train      = 180        # total training patches (split equally across classes)
random_seed  = 42
image_folder = r"D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM"
train_cache  = r"Results\conch_train_features_fewshot.pt"   # cached features (reuse if exists)
clf_save_path = rf"Results\Few_Shot_Evaluation\{n_train}_trainset\few_shot_logistic_regression.pkl"

os.makedirs("Results", exist_ok=True)

# ========== LOAD CLASSES ==========
with open(prompt_file) as f:
    prompt_data = json.load(f)["0"]
classes_list = list(prompt_data["classnames"].keys())
print(f"Classes: {classes_list}")

# ========== BUILD TRAIN DATAFRAME ==========
extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
records = []
for cls in classes_list:
    cls_dir = os.path.join(image_folder, cls)
    if not os.path.isdir(cls_dir):
        print(f"[WARNING] Missing folder: {cls_dir}")
        continue
    for fname in os.listdir(cls_dir):
        if Path(fname).suffix.lower() in extensions:
            records.append({"patch_name": fname, "label": cls,
                            "full_path": os.path.join(cls_dir, fname)})

import pandas as pd
full_df = pd.DataFrame(records)
print(f"Total patches found: {len(full_df)}")

# Class-balanced sample for training
rng = np.random.default_rng(random_seed)
per_class = n_train // len(classes_list)
train_indices = []
for cls in classes_list:
    idx = full_df.index[full_df["label"] == cls].tolist()
    rng.shuffle(idx)
    train_indices.extend(idx[:min(per_class, len(idx))])

train_df = full_df.loc[train_indices].reset_index(drop=True)
print(f"Training on {len(train_df)} patches ({per_class} per class)")

# ========== EXTRACT / LOAD FEATURES ==========
# model and preprocess must already be loaded (from the cells above in the notebook)

if os.path.exists(train_cache):
    print(f"Loading cached features from {train_cache} ...")
    cache = torch.load(train_cache, weights_only=False)
    X_train, y_train = cache["features"].numpy(), cache["labels"]
else:
    print("Extracting features...")
    features, labels = [], []
    with torch.inference_mode():
        for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
            try:
                img = Image.open(row["full_path"]).convert("RGB")
                t = preprocess(img).unsqueeze(0).to(device)
                feat = model.encode_image(t)
                feat = feat / feat.norm(dim=-1, keepdim=True)
                features.append(feat.cpu().squeeze(0))
                labels.append(row["label"])
            except Exception as e:
                print(f"[WARN] {row['full_path']}: {e}")

    X_train_t = torch.stack(features)
    y_train   = np.array(labels)
    torch.save({"features": X_train_t, "labels": y_train}, train_cache)
    print(f"Features cached → {train_cache}")
    X_train = X_train_t.numpy()

# ========== TRAIN & SAVE ==========
clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_seed)
clf.fit(X_train, y_train)
joblib.dump(clf, clf_save_path)
print(f"Linear probe saved → {clf_save_path}")

Classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
Total patches found: 100000
Training on 180 patches (20 per class)
Extracting features...


  0%|          | 0/180 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
100%|██████████| 180/180 [00:03<00:00, 47.36it/s]

Features cached → Results\conch_train_features_fewshot.pt
Linear probe saved → Results\Few_Shot_Evaluation\180_trainset\few_shot_logistic_regression.pkl


#### Below script will create a csv file that matches each patch to 1 caption

In [3]:
import pandas as pd
import os
import joblib
from tqdm import tqdm
import torch
from PIL import Image

# USER INPUTS
main_folder = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data"

# Path to the saved logistic regression model from Few_Shot_label_matching
clf_path = r"Results\Few_Shot_Evaluation\180_trainset\few_shot_logistic_regression.pkl"

# OPTIONAL → set to None if you want to process ALL images
non_white_csv = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv"

output_csv = r"Results\classification_results_tcga_fewshot.csv"

# Load the saved linear probe classifier
clf = joblib.load(clf_path)
print(f"Loaded linear probe classifier from {clf_path}")

# LOAD CSV IF PROVIDED
patch_names_to_process = None
if non_white_csv and os.path.exists(non_white_csv):
    print("Using non-white CSV filter...")
    non_white_df = pd.read_csv(non_white_csv)
    patch_names_to_process = set(non_white_df["patch_name"].tolist())
else:
    print("No non-white CSV provided → processing ALL images")

# STORAGE
csv_data = []

# COLLECT ALL IMAGES (RECURSIVE)
all_image_files = []
for root, dirs, files in os.walk(main_folder):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg", ".tif")):
            all_image_files.append((root, file))

print(f"Found {len(all_image_files)} total image files")

# OPTIONAL FILTERING
if patch_names_to_process is not None:
    files_to_process = [
        (root, file)
        for root, file in all_image_files
        if file in patch_names_to_process
    ]
    print(f"Processing {len(files_to_process)} files based on CSV filter")
else:
    files_to_process = all_image_files
    print(f"Processing ALL {len(files_to_process)} files")

# REMOVE OLD CSV
if os.path.exists(output_csv):
    os.remove(output_csv)

# PROCESS IMAGES — Few-Shot Linear Probe
with torch.inference_mode():
    for i, (root, img_name) in enumerate(tqdm(files_to_process, desc="Processing images")):
        img_path = os.path.join(root, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
            image_tensor = preprocess(image).unsqueeze(0).to(device)

            # Extract L2-normalised features (same as training the probe)
            feat = model.encode_image(image_tensor)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            feat_np = feat.cpu().numpy()

            # Linear probe prediction
            pred_label = clf.predict(feat_np)[0]
            pred_score = float(clf.predict_proba(feat_np).max())

            csv_data.append({
                "patch_name": img_name,
                "label": pred_label,
                "score": pred_score
            })

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

        # SAVE EVERY 10K
        if (i + 1) % 10000 == 0:
            df = pd.DataFrame(csv_data)
            write_header = not os.path.exists(output_csv)
            df.to_csv(output_csv, mode="a", header=write_header, index=False)
            csv_data = []
            print(f"Saved checkpoint at {i+1} images")

if csv_data:
    df = pd.DataFrame(csv_data)
    write_header = not os.path.exists(output_csv)
    df.to_csv(output_csv, mode="a", header=write_header, index=False)

print(f"\nResults saved to '{output_csv}'")
print(f"Total patches processed: {len(files_to_process)}")

Loaded linear probe classifier from Results\Few_Shot_Evaluation\180_trainset\few_shot_logistic_regression.pkl
Using non-white CSV filter...
Found 1460238 total image files
Processing 1377933 files based on CSV filter


Processing images:   0%|          | 0/1377933 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
Processing images:   1%|          | 10004/1377933 [05:50<14:47:23, 25.69it/s]

Saved checkpoint at 10000 images


Processing images:   1%|▏         | 20002/1377933 [12:10<14:54:12, 25.31it/s]

Saved checkpoint at 20000 images


...

Saved checkpoint at 1360000 images


Processing images:  99%|█████████▉| 1370002/1377933 [14:00:16<05:26, 24.31it/s]

Saved checkpoint at 1370000 images


Processing images: 100%|██████████| 1377933/1377933 [14:05:32<00:00, 27.16it/s]


Results saved to 'Results\classification_results_tcga_fewshot.csv'
Total patches processed: 1377933


### Validate number of entries in csv

In [ ]:
import csv

def count_entries(csv_file):
    with open(csv_file, newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        # Count the number of rows in the CSV file
        row_count = sum(1 for row in reader)
    return row_count

# Example usage
csv_file = r'Results\classification_results_crc100k.csv'
print(f"Number of entries: {count_entries(csv_file)}")